### 自动微分（Automatic Differentiation）

1. 深度学习框架可以通过 **自动微分**（*automatic differentiation*）来加快求导。
2. 系统会根据模型构建 **计算图**（*computational graph*），跟踪计算是 *哪些数据* 经 *哪些操作* 组合产生输出。
3. **反向传播**（*backpropagation*）意味着跟踪整个计算图，填充 **每个参数的偏导数**（*partial derivative*）。


In [35]:
import torch
x = torch.arange(4.0)


用以下代码 **防止不断分配新内存**（in-place 原地操作）：即将 `x` 标记为 **需要梯度**（*requires grad*）


In [ ]:
x.requires_grad_(True) 
x.grad

**注意！！** 最后带 `_` 的才是原地操作的 **方法**，而 `.requires_grad(True)` 仅仅是一个属性

In [37]:
y = 2 * torch.dot(x, x) #一维张量的点积
y

tensor(28., grad_fn=<MulBackward0>)

通过调用 **反向传播函数** (*backpropagate*) 来自动计算 `y` 对于 `x` 每个分量的梯度

In [38]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

- **注意**，每一个刚刚出生的 `y` 只能反向传播一次，因为每次反向传播后都会释放计算图以节约内存。  
- 若重复执行两次上方的代码块则会报错

下面需要将x的梯度清零，因为pytorch会**积累梯度**

In [39]:
x.grad.zero_()
y=x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

- **注意** `x.grad` 是将x作为未知量求y的偏导后，再将x的具体值带入得到的，而不是对数值的操作

***
下面讨论 **非标量（即向量）对向量的微分** ，得到的结果是 **矩阵** ，结果按照 **分母顺序** 排序

同理，对于高维的 `x` 与 `y` , 得到的结果将是 **高阶张量**

In [40]:
x.grad.zero_()
y = x*x
y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

- 实际上，pytorch对向量的操作会将 **雅可比矩阵** 降维成一个 **向量**
***

#### 分离计算

有时候我们希望把某些计算从 **计算图** 中移出，比如 `z` 是 关于 `x` 与 `y` 的一个函数，而希望把 `y` 当作一个常数来求关于 `x` 的 **偏导** ，则用 `u` 替换 `y`，以防止对 `y` 的计算的记录

In [45]:
x.grad.zero_()
y = torch.arange(1.0,5.0)
y.requires_grad_(True)
z = y*x
print(z)
z.sum().backward() #只记录了关于x的计算，故不会求关于y的偏导
print(x.grad)
print(y.grad)

tensor([ 0.,  2.,  6., 12.], grad_fn=<MulBackward0>)
tensor([1., 2., 3., 4.])
tensor([0., 1., 2., 3.])
